In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import wandb
from scipy.stats import gmean

In [ ]:
# Results come from Weights & Biases. Point these at the project the
# experiments were logged to; the server address and credentials come from the
# usual wandb configuration (the WANDB_BASE_URL environment variable or
# ~/.config/wandb/settings).
WANDB_PROJECT = "fair-irl"
WANDB_ENTITY = None  # None uses the default entity of your wandb login

# Every run of one execution of Fair_IRL_Biased_Demonstrations.py shares a
# SESSION_ID. Only one session is ever loaded, so a plot can never mix results
# from two executions of the training script. Leave this None to use the most
# recent execution, or set it to a session id (e.g. "20260826-010125-3f9ab2")
# to reproduce an earlier set of figures exactly.
WANDB_SESSION = None


def to_tuple(data):
    if isinstance(data, list):
        return tuple(to_tuple(i) for i in data)
    elif isinstance(data, dict):
        return {k: to_tuple(v) for k, v in data.items()}
    else:
        return data


class ResultsLookup(dict):
    """Nested results dict that names the missing configuration on a miss.

    The plotting cells below index five levels deep, e.g.
    `averaged_data[dataset][expert][algorithm][dataset_bias_type][weight_adjust]`.
    With a plain defaultdict a configuration the session never ran would
    silently produce an empty frame, and the failure only surfaced later as a
    confusing
    `KeyError: 'sum_abs_subdominance_val'` from `.loc`. This reports which
    configuration is actually absent, and what the session does contain.
    """

    LEVELS = ("dataset", "expert", "algorithm", "dataset_bias_type", "weight_adjust")

    def __init__(self, depth=0, path=(), session_id=""):
        super().__init__()
        self._depth = depth
        self._path = path
        self._session_id = session_id

    def child(self, key):
        """Get or create the sub-lookup for `key` (never triggers __missing__)."""
        return self.setdefault(
            key,
            ResultsLookup(self._depth + 1, self._path + (key,), self._session_id),
        )

    def __missing__(self, key):
        level = self.LEVELS[self._depth]
        context = ", ".join(f"{n}={v!r}" for n, v in zip(self.LEVELS, self._path))
        raise KeyError(
            f"No results for {level}={key!r}"
            + (f" under {context}" if context else "")
            + f" in W&B session {self._session_id!r}."
            f" Available {level} values there: {sorted(self.keys(), key=repr)}."
            " Either re-run the training script with a config that produces it,"
            " or set WANDB_SESSION to a session that has it."
        )


def resolve_session(api, path, session=None):
    """Return the session id to load: `session`, or the most recent one."""
    if session is not None:
        return session

    newest = next(iter(api.runs(path, order="-created_at", per_page=1)), None)
    if newest is None:
        raise RuntimeError(
            f"No runs found in W&B project {path!r}."
            " Run Fair_IRL_Biased_Demonstrations.py first."
        )
    return newest.config["SESSION_ID"]


def read_session_runs(project, entity=None, session=None):
    """Read the runs of a single execution of the training script.

    Each run is one trial of one algorithm, one dataset bias type and one
    weight adjustment. Its config holds the experiment parameters and its
    summary holds that trial's results.

    Returns
    -------
    session_id : str
        The session that was loaded.
    records : list<dict>
        One record per run, with its `created_at`, `config`, `metrics`,
        `expert_demo_losses` and `superhuman_demo_losses`.
    """
    api = wandb.Api()
    path = f"{entity}/{project}" if entity else project
    session_id = resolve_session(api, path, session)

    records = []
    for run in api.runs(
        path, filters={"state": "finished", "config.SESSION_ID": session_id}
    ):
        summary = dict(run.summary)
        # Trials that did not converge have no results to plot.
        if not summary.get("converged", False):
            continue

        # Config values come back from W&B as lists; the plotting cells index
        # with tuple literals, so they have to be tuples to be usable as keys.
        config = to_tuple(
            {k: v for k, v in run.config.items() if not k.startswith("_")}
        )
        metrics = {
            k: v
            for k, v in summary.items()
            if not k.startswith("_")
            and isinstance(v, (int, float))
            and not isinstance(v, bool)
        }

        # The expert's per-demonstration losses are lists (one row per
        # subdominance group), not scalars, so the filter above drops them. They
        # are kept apart: they have no meaningful average across trials.
        expert_demo_losses = {
            k: v for k, v in summary.items() if k.startswith("expert_demo_feat_loss_")
        }
        # Likewise the demonstrations the Superhuman Fairness model imitated,
        # which only its own runs record.
        superhuman_demo_losses = {
            k: v
            for k, v in summary.items()
            if k.startswith("superhuman_demo_feat_loss")
        }

        records.append(
            {
                "created_at": run.created_at,
                "config": config,
                "metrics": metrics,
                "expert_demo_losses": expert_demo_losses,
                "superhuman_demo_losses": superhuman_demo_losses,
            }
        )

    return session_id, records


session_id, records = read_session_runs(WANDB_PROJECT, WANDB_ENTITY, WANDB_SESSION)
if not records:
    raise RuntimeError(
        f"W&B session {session_id!r} has no finished, converged runs."
        " Pick another session with WANDB_SESSION, or re-run the training script."
    )

# Collect the trials of each (dataset, expert, algorithm, dataset bias type,
# weight adjustment) configuration within this one session. Each run covers
# exactly one algorithm, one bias type and one weight adjustment, so the latter
# two are single tuples -- `()` for the unbiased dataset and for the unadjusted
# weights. Sessions logged before the Superhuman Fairness baseline existed have
# no ALGORITHM in their config; those runs are all FairIRL Bias Reduction ones.
trials_by_config = defaultdict(list)
trial_ids_by_config = defaultdict(list)
config_by_key = {}
first_seen = {}
# The expert's demonstrations, as losses per subdominance group, keyed by
# (dataset, expert, dataset bias type, trial). Every algorithm of one bias type
# and trial is scored against the same groups, so any one of their runs serves.
# Read them through `get_expert_demo_losses()` below.
expert_demo_losses = {}
# The demonstrations the Superhuman Fairness model imitated, keyed the same way.
# Read them through `get_superhuman_demo_losses()` below.
superhuman_demo_losses = {}
for record in records:
    config = record["config"]
    config_key = (
        config["DATASET"],
        config["EXPERT_ALGO"],
        config.get("ALGORITHM", "FairIRL Bias Reduction"),
        config["DATASET_BIAS_TYPE"],
        config["WEIGHT_ADJUST"],
    )
    trials_by_config[config_key].append(record["metrics"])
    trial_ids_by_config[config_key].append(config.get("TRIAL", 0))
    config_by_key[config_key] = config
    if record["expert_demo_losses"]:
        demo_key = (
            config["DATASET"],
            config["EXPERT_ALGO"],
            config["DATASET_BIAS_TYPE"],
            config.get("TRIAL", 0),
        )
        expert_demo_losses.setdefault(demo_key, record["expert_demo_losses"])
    if record["superhuman_demo_losses"]:
        superhuman_demo_losses[
            (
                config["DATASET"],
                config["EXPERT_ALGO"],
                config["DATASET_BIAS_TYPE"],
                config.get("TRIAL", 0),
            )
        ] = record["superhuman_demo_losses"]
    first_seen[config_key] = min(
        first_seen.get(config_key, record["created_at"]), record["created_at"]
    )

# Average across all trials of each configuration. `per_trial_data` keeps the
# same results un-averaged -- one row per trial, indexed by TRIAL -- for plots
# that need the spread across trials (e.g. error bars).
averaged_data = ResultsLookup(session_id=session_id)
per_trial_data = ResultsLookup(session_id=session_id)
averaged_info = {}
# Oldest configuration first, so that datasets appear in the order they were
# run, which is the order the plots lay them out along the x-axis.
for config_key in sorted(trials_by_config, key=lambda k: first_seen[k]):
    dataset, expert, algorithm, dataset_bias_type, weight_adjust = config_key
    trial_data = pd.DataFrame(
        trials_by_config[config_key],
        index=pd.Index(trial_ids_by_config[config_key], name="TRIAL"),
    )
    averaged_data.child(dataset).child(expert).child(algorithm).child(
        dataset_bias_type
    )[weight_adjust] = trial_data.mean()
    per_trial_data.child(dataset).child(expert).child(algorithm).child(
        dataset_bias_type
    )[weight_adjust] = trial_data
    averaged_info.setdefault(dataset, {}).setdefault(expert, {}).setdefault(
        algorithm, {}
    ).setdefault(dataset_bias_type, {})[weight_adjust] = config_by_key[config_key]

print(f"W&B session:  {session_id}")
print(f"runs loaded:  {len(records)} across {len(trials_by_config)} configurations")
for config_key in sorted(trials_by_config, key=lambda k: first_seen[k]):
    dataset, expert, algorithm, dataset_bias_type, weight_adjust = config_key
    n = len(trials_by_config[config_key])
    print(
        f"  {dataset} / {expert} / {algorithm} / bias={dataset_bias_type}"
        f" / weights={weight_adjust}: {n} trial(s)"
    )


def get_expert_demo_losses(dataset, expert, dataset_bias_type, trial=0, split="train"):
    """The expert's demonstrations of one trial, as losses per subdominance group.

    Returns
    -------
    losses : pandas.DataFrame
        One row per subdominance group (i.e. per demonstration), one column per
        subdominance metric, holding that metric as a loss (lower is better).
    """
    key = (dataset, expert, dataset_bias_type, trial)
    if key not in expert_demo_losses:
        raise KeyError(
            f"No expert demonstration losses for dataset={dataset!r},"
            f" expert={expert!r}, dataset_bias_type={dataset_bias_type!r},"
            f" trial={trial!r} in W&B session {session_id!r}. Sessions logged"
            " before these were recorded do not have them; re-run the training"
            " script. Available: "
            f"{sorted(expert_demo_losses, key=repr)}."
        )
    stored = expert_demo_losses[key]
    return pd.DataFrame(
        np.asarray(stored[f"expert_demo_feat_loss_{split}"], dtype=float),
        columns=list(stored["expert_demo_feat_loss_metrics"]),
    )


def get_superhuman_demo_losses(dataset, expert, dataset_bias_type, trial=0):
    """The demonstrations the Superhuman Fairness model of one trial imitated.

    With SH_DEMO_SOURCE "pp_baseline" these are the original paper's
    `post_proc_demos`; with "expert_demos" they are the expert's training
    subdominance groups.

    Returns
    -------
    losses : pandas.DataFrame
        One row per demonstration, one column per Superhuman Fairness feature
        (its SH_FEATURES), holding that feature as a loss (lower is better).
    """
    key = (dataset, expert, dataset_bias_type, trial)
    if key not in superhuman_demo_losses:
        raise KeyError(
            f"No Superhuman Fairness demonstration losses for dataset={dataset!r},"
            f" expert={expert!r}, dataset_bias_type={dataset_bias_type!r},"
            f" trial={trial!r} in W&B session {session_id!r}. Sessions logged"
            " before these were recorded do not have them; re-run the training"
            " script. Available: "
            f"{sorted(superhuman_demo_losses, key=repr)}."
        )
    stored = superhuman_demo_losses[key]
    return pd.DataFrame(
        np.asarray(stored["superhuman_demo_feat_loss"], dtype=float),
        columns=list(stored["superhuman_demo_feat_loss_metrics"]),
    )


# Keep the notebook namespace clean for the plotting cells below. Done by
# name so that it stays correct however many of these were actually bound.
for _name in (
    "records",
    "record",
    "config",
    "trials_by_config",
    "trial_ids_by_config",
    "config_by_key",
    "first_seen",
    "config_key",
    "demo_key",
    "trial_data",
    "dataset",
    "expert",
    "algorithm",
    "dataset_bias_type",
    "weight_adjust",
):
    globals().pop(_name, None)
del _name

In [ ]:
selected_expert = "PostProcDemo"

# Which technique the plots below are drawn for. Every run carries its
# technique as `ALGORITHM`, so switching this re-draws the same figures for
# another one; a comparison plot can index several, e.g.
# `averaged_data[dataset][selected_expert]["Superhuman Fairness"][bias][()]`.
# Valid values are the entries of the training script's ALGORITHMS list:
# "FairIRL Bias Reduction", "Superhuman Fairness", "Post Proc DP",
# "Post Proc EqOdds", "Fair LogLoss DP" and "Fair LogLoss EqOdds".
selected_algorithm = "FairIRL Bias Reduction"

# Each W&B run covers exactly one dataset bias type and one weight adjustment,
# so every key below is a single tuple, matching one entry of the training
# script's DATASET_BIAS_TYPE_LIST / WEIGHT_ADJUST_LIST -- plus the `()` that is
# always run for the unbiased dataset and for the unadjusted weights. Only
# configurations the loaded session actually ran can be selected here; every
# technique other than FairIRL Bias Reduction has no reward weights to adjust,
# so those only ever report the unadjusted `()` one.
unbiased_types = [
    (),
]
biased_types = [
    # ("unbalanced_redlining", 0.2),
    # ("balanced_redlining", 0.2),
    # ("perfectly_balanced_redlining", 0.2),
    # ("corruption_bias", "CatBoost", 0.001, "gaussian", 1.0),
]

unadjusted_weights = ()
adjusted_weights = (
    # ("mul_negative_weights", 0.0),
    # ("mul_negative_weights", 0.1),
    # ("mul_negative_weights", 0.2),
    # ("mul_negative_weights", 0.3),
    # ("mul_negative_weights", 0.4),
    # ("mul_negative_weights", 0.5),
    # ("mul_negative_weights", 0.6),
    # ("mul_negative_weights", 0.7),
    # ("mul_negative_weights", 0.8),
    # ("mul_negative_weights", 0.9),
    # ("opt_debias", "optuna", "CMA-ES", 500),
    # ("opt_debias", "pybobyqa", "Multi-Start BOBYQA", 500),
    ("opt_debias", "nevergrad", "BayesOpt", 200),
    # ("opt_debias", "nevergrad", "Nelder-Mead", 500),
    # ("opt_debias", "nevergrad", "Powell", 500),
)

In [ ]:
# Shared setup for the two Superhuman Fairness figures below, `plot_features`
# and `plot_features_errorbars`, both migrated from `plot.py` of the Superhuman
# Fairness reference implementation
# (https://github.com/omidMemari/superhumn-fairness).
#
# Both draw, for every pair of performance/fairness losses, one figure per
# dataset, expert and dataset bias type: the expert's demonstrations, the
# Superhuman Fairness model on its training split -- with the 1/alpha margins it
# is trained to beat the demonstrations by -- and on the test split, every
# fair-classification baseline on the test split, and the FairIRL Bias
# Reduction model on the test split. Every value is a loss: lower is better.
#
# They are structured like the originals, with these changes so that they fit
# this project:
# * The losses are this project's Acc (as prediction error) and the
#   differences of AccPar, DemPar, EqOpp and TNRPar, in place of the paper's
#   error, DP, EqOdds and PRP. As in the original, the error is listed first
#   and so is only ever on the y axis: every x axis is one of the four
#   differences.
# * Every dataset, expert and dataset bias type of the loaded W&B session is
#   plotted, instead of the paper's Adult and COMPAS (the bias types play the
#   role of the original's noise settings).
# * By default the scattered demonstrations are the expert's (the EXPERT_ALGO
#   each run was configured with) -- one star per subdominance group -- instead
#   of post-processing ones. `plot_demo_source = "superhuman"` scatters the
#   demonstrations the Superhuman Fairness model actually imitated instead,
#   which are the original's `post_proc_demos` when SH_DEMO_SOURCE is
#   "pp_baseline".
# * The FairIRL Bias Reduction model is added, drawn in the style the original
#   gave MFOpt, which is left out because no implementation of it exists. It is
#   left out of a figure whose configuration the session did not run it for.
# * The fair log-loss models are drawn at their measured losses, unless
#   `fair_logloss_paper_metrics` is set; see `fair_logloss_losses()`.
#
# To reproduce the paper's figures, train on the Adult_SH/COMPAS_SH datasets
# (which the training script runs under the paper's conditions) and set
# `plot_demo_source = "superhuman"` and `fair_logloss_paper_metrics = True`.
# * The original also read a fair log-loss EqOpp baseline, but only for a line
#   it had commented out; this project has no such baseline.
# * Figures are shown inline as well as saved.
import os

# The losses to plot, in order. Each must be one of the
# SUBDOMINANCE_*_METRICS_LIST the session ran with, since those are the
# measures both the expert's demonstrations and the models are recorded on.
# Those may be named as this project's objectives ("Acc", "DemPar", ...) or as
# the original paper's metrics ("inacc", "dp", "prp", ...). None plots every
# metric of the session, in order.
feature_list = None
# The dataset bias types to plot. None plots every one the session ran.
plot_dataset_bias_types = None
# The FairIRL Bias Reduction weight adjustment to plot. None uses the session's
# only non-empty WEIGHT_ADJUST (or the unadjusted weights if it has none).
plot_fairirl_weight_adjust = None
# Which demonstrations are scattered, and the gamma-superhuman diagnostics
# computed against:
#   "expert"     -- the expert's training subdominance groups.
#   "superhuman" -- the demonstrations the Superhuman Fairness model imitated,
#                   the original `plot_features()`'s `post_proc_demos` when its
#                   SH_DEMO_SOURCE is "pp_baseline".
plot_demo_source = "expert"
# The original plots the fair log-loss baselines at their own expected zero-one
# loss and their own fairness violation, in place of the measured prediction
# error and DP/EqOdds difference. True does the same; False plots every model
# at its measured losses.
fair_logloss_paper_metrics = False

plots_path = "./../../experiment_output/fair_irl/exp_plots/superhuman_features"
os.makedirs(plots_path, exist_ok=True)

# The metrics the loaded session actually recorded, in its own order.
session_metric_names = list(
    next(iter(expert_demo_losses.values()))["expert_demo_feat_loss_metrics"]
)
if feature_list is None:
    feature_list = session_metric_names
missing_metrics = [f for f in feature_list if f not in session_metric_names]
if missing_metrics:
    raise ValueError(
        f"feature_list names {missing_metrics}, which W&B session {session_id!r}"
        f" did not record. It ran with {session_metric_names}; plot a subset of"
        " those, or re-run the training script with the metrics you want in"
        " SUBDOMINANCE_PERF_METRICS_LIST / SUBDOMINANCE_FAIR_METRICS_LIST."
    )

feature = dict(enumerate(feature_list))
num_of_features = len(feature_list)

# Axis labels, in the style of the original's `name` and `short` dicts. A
# metric with no entry falls back to its own name.
name = {
    "Acc": "Prediction error",
    "AccPar": "D.AccPar",
    "DemPar": "D.DP",
    "EqOpp": "D.EqOpp",
    "TNRPar": "D.TNR",
    "FPRPar": "D.FPR",
    "FNRPar": "D.FNR",
    "EqOdds": "D.EqOdds",
    "PredPar": "D.PPV",
    "NegPredPar": "D.NPV",
    "inacc": "Prediction error",
    "dp": "D.DP",
    "eqodds": "D.EqOdds",
    "prp": "D.PRP",
    "eqopp": "D.FNR",
    "fnr": "D.FNR",
    "fpr": "D.FPR",
    "ppv": "D.PPV",
    "npv": "D.NPV",
    "error_rate_diff": "D.Balanced Error Rate",
}
short = {
    "Acc": "error",
    "AccPar": "AccPar",
    "DemPar": "DP",
    "EqOpp": "EqOpp",
    "TNRPar": "TNR",
    "FPRPar": "FPR",
    "FNRPar": "FNR",
    "EqOdds": "EqOdds",
    "PredPar": "PPV",
    "NegPredPar": "NPV",
    "inacc": "error",
    "dp": "DP",
    "eqodds": "EqOdds",
    "prp": "PRP",
    "eqopp": "FNR",
    "fnr": "FNR",
    "fpr": "FPR",
    "ppv": "PPV",
    "npv": "NPV",
    "error_rate_diff": "D.ErrorRate",
}
name = {**{f: f for f in feature_list}, **name}
short = {**{f: f for f in feature_list}, **short}


def feature_losses(results, split):
    """A model's loss on every plotted feature, from its W&B results.

    `results` is one configuration's entry of `averaged_data` (a Series) or of
    `per_trial_data` (a DataFrame, giving one loss per trial).

    `model_feat_loss_*` holds the model's loss on each subdominance metric
    directly, whichever vocabulary names it. Sessions logged before those were
    recorded fall back to the feature expectations, which are "goodness"
    measures, so a loss is `1 - mu` -- the same inversion the expert's
    demonstration losses were recorded with.
    """
    available = results.index if isinstance(results, pd.Series) else results.columns
    losses = {}
    for f in feature_list:
        if f"model_feat_loss_{split}_{f}" in available:
            losses[f] = results[f"model_feat_loss_{split}_{f}"]
            continue
        for key in (f"muL_perf_{split}_{f}", f"muL_{split}_{f}"):
            if key in available:
                losses[f] = 1 - results[key]
                break
        else:
            raise KeyError(
                f"No {split} loss for {f!r} in the results. Sessions logged"
                " before model_feat_loss_* was recorded only cover metrics named"
                " as this project's objectives; re-run the training script to"
                " plot the original paper's metrics."
            )
    return losses


def bias_type_label(dataset_bias_type):
    """A filename-friendly name for a dataset bias type."""
    if not dataset_bias_type:
        return "unbiased"
    return "_".join(str(component) for component in dataset_bias_type)


def resolve_fairirl_weight_adjust(dataset, expert, dataset_bias_type, requested):
    """The FairIRL Bias Reduction weight adjustment to plot, or None when the
    session did not run FairIRL Bias Reduction for this configuration (the
    figures then leave it out)."""
    fairirl_results = averaged_data[dataset][expert].get("FairIRL Bias Reduction", {})
    if dataset_bias_type not in fairirl_results:
        return None
    if requested is not None:
        return requested
    available = list(fairirl_results[dataset_bias_type])
    adjusted = [w for w in available if w != ()]
    if len(adjusted) > 1:
        raise ValueError(
            f"The session ran several FairIRL weight adjustments ({adjusted});"
            " set plot_fairirl_weight_adjust to the one to plot."
        )
    return adjusted[0] if adjusted else ()


def get_plot_demo_losses(dataset, expert, dataset_bias_type, trial):
    """The demonstrations to scatter, per `plot_demo_source`, as losses on the
    plotted features, and their legend label."""
    if plot_demo_source == "expert":
        losses = get_expert_demo_losses(
            dataset, expert, dataset_bias_type, trial=trial, split="train"
        )
        return losses, f"{expert}_demos"
    if plot_demo_source == "superhuman":
        losses = get_superhuman_demo_losses(
            dataset, expert, dataset_bias_type, trial=trial
        )
        missing = [f for f in feature_list if f not in losses.columns]
        if missing:
            raise KeyError(
                f"The Superhuman Fairness demonstrations have no loss for"
                f" {missing}; they were scored on {list(losses.columns)}. Its"
                " SH_FEATURES must include every plotted feature."
            )
        sh_config = averaged_info[dataset][expert]["Superhuman Fairness"][
            dataset_bias_type
        ][()]
        if sh_config.get("SH_DEMO_SOURCE_RESOLVED") == "pp_baseline":
            label = "post_proc_demos"
        else:
            label = f"{expert}_demos"
        return losses[feature_list], label
    raise ValueError(
        f"Unrecognized plot_demo_source: {plot_demo_source!r}."
        " Valid values are 'expert' and 'superhuman'."
    )


# The feature each fair log-loss model's own fairness violation stands in for,
# by the model's fairness constraint.
FAIR_LOGLOSS_VIOLATION_FEATURE = {
    "demographic_parity": "dp",
    "equalized_odds": "eqodds",
}


def fair_logloss_losses(results, split, constraint):
    """A fair log-loss model's losses, as `feature_losses()` returns them.

    With `fair_logloss_paper_metrics`, the model's own expected zero-one loss
    stands in for "inacc", and its own fairness violation for the difference
    of its constraint ("dp" or "eqodds"), as `eval_model_baseline()` of the
    original reports them "since fair logloss uses expected violation".
    """
    losses = feature_losses(results, split)
    if not fair_logloss_paper_metrics:
        return losses
    if "inacc" in losses:
        losses["inacc"] = results[f"fair_logloss_expected_zeroone_{split}"]
    violation_feature = FAIR_LOGLOSS_VIOLATION_FEATURE[constraint]
    if violation_feature in losses:
        losses[violation_feature] = results[f"fair_logloss_violation_{split}"]
    return losses


def superhuman_alpha(results):
    """The Superhuman Fairness model's final alpha (`model_params["alpha"]`),
    for the plotted features, in order."""
    missing = [f for f in feature_list if f"superhuman_alpha_{f}" not in results.index]
    if missing:
        raise KeyError(
            f"The Superhuman Fairness run has no alpha for {missing}. Its"
            " SH_FEATURES must include every plotted feature, and the session must"
            " have been logged after superhuman_alpha_* was recorded."
        )
    return [float(results[f"superhuman_alpha_{f}"]) for f in feature_list]


def find_gamma_superhuman(demo_losses, model_loss):
    """`util.find_gamma_superhuman()`: per feature, the fraction of demos the
    model matches or beats."""
    print("gamma-superhuman: ")
    gamma_superhuman_arr = []
    for i in range(num_of_features):
        demo_loss = demo_losses[feature[i]].tolist()
        f = feature[i]
        n = len(demo_loss)
        count = 0
        for j in range(n):
            if model_loss[f] <= demo_loss[j]:
                count += 1
        gamma_superhuman = count / n
        print(gamma_superhuman, f)
        gamma_superhuman_arr.append(gamma_superhuman)
    return gamma_superhuman_arr


def find_gamma_superhuman_all(demo_losses, method_losses):
    """`util.find_gamma_superhuman_all()`: per model, the fraction of demos it
    matches or beats on every feature at once."""
    print("gamma-superhuman: ")
    baseline = dict(enumerate(method_losses))
    baseline_loss = np.zeros(len(baseline))
    dominated = np.zeros(len(baseline))
    for j in range(len(demo_losses)):
        count_baseline = np.zeros(len(baseline))
        for i in range(num_of_features):
            demo_loss = demo_losses[feature[i]].iloc[j]
            for k in range(len(baseline)):
                baseline_loss[k] = method_losses[baseline[k]][feature[i]]
            for k in range(len(baseline)):
                if baseline_loss[k] <= demo_loss:
                    count_baseline[k] += 1
                    if count_baseline[k] == num_of_features:
                        dominated[k] += 1
    dominated = dominated / len(demo_losses)
    print(baseline)
    print("dominated:")
    print(dominated)


def plotted_configurations():
    """Every (dataset, expert, dataset bias type) of the session to plot."""
    for dataset in averaged_info:
        for expert in averaged_info[dataset]:
            bias_types = plot_dataset_bias_types
            if bias_types is None:
                bias_types = list(averaged_data[dataset][expert]["Superhuman Fairness"])
            for dataset_bias_type in bias_types:
                yield dataset, expert, dataset_bias_type

In [ ]:
# plot_features: `plot_features()` of the Superhuman Fairness reference
# implementation's `plot.py`. Run the setup cell above first.
#
# The model points are the means over the session's trials (`averaged_data`),
# and the demonstrations are those of trial `demo_trial`, since the original
# plotted one experiment's demonstrations.
demo_trial = 0


def plot_features(dataset, expert, dataset_bias_type, fairirl_weight_adjust):
    by_algorithm = averaged_data[dataset][expert]
    sh_results = by_algorithm["Superhuman Fairness"][dataset_bias_type][()]

    demo_losses, demo_label = get_plot_demo_losses(
        dataset, expert, dataset_bias_type, trial=demo_trial
    )
    print(dataset, expert, bias_type_label(dataset_bias_type))
    alpha = superhuman_alpha(sh_results)
    print("alpha: ", alpha)
    alpha = [1 / x for x in alpha]
    print(alpha)

    ### our model, on its training split and on the test split
    eval_train = feature_losses(sh_results, "train")
    eval_sh = feature_losses(sh_results, "test")
    ### post-processing
    eval_pp_dp = feature_losses(by_algorithm["Post Proc DP"][dataset_bias_type][()], "test")
    eval_pp_eq_odds = feature_losses(by_algorithm["Post Proc EqOdds"][dataset_bias_type][()], "test")
    ### fair log-loss
    eval_fairll_dp = fair_logloss_losses(by_algorithm["Fair LogLoss DP"][dataset_bias_type][()], "test", "demographic_parity")
    eval_fairll_eqodds = fair_logloss_losses(by_algorithm["Fair LogLoss EqOdds"][dataset_bias_type][()], "test", "equalized_odds")
    ### FairIRL Bias Reduction, if the session ran it for this configuration
    eval_fair_irl = None
    if fairirl_weight_adjust is not None:
        eval_fair_irl = feature_losses(
            by_algorithm["FairIRL Bias Reduction"][dataset_bias_type][fairirl_weight_adjust],
            "test",
        )

    find_gamma_superhuman(demo_losses, eval_train)
    method_losses = {
        "eval_pp_dp": eval_pp_dp,
        "eval_pp_eq_odds": eval_pp_eq_odds,
        "eval_fairll_dp": eval_fairll_dp,
        "eval_fairll_eqodds": eval_fairll_eqodds,
        "eval_fair_irl": eval_fair_irl,
        "superhuman": eval_train,
    }
    find_gamma_superhuman_all(
        demo_losses,
        {method: losses for method, losses in method_losses.items() if losses is not None},
    )

    for i in range(num_of_features):
        for j in range(i + 1, num_of_features):
            demo_metric_i = demo_losses[feature[i]].tolist()
            demo_metric_j = demo_losses[feature[j]].tolist()
            f1 = plt.figure()
            ### our model
            x = eval_train[feature[j]]
            y = eval_train[feature[i]]
            x_test = eval_sh[feature[j]]
            y_test = eval_sh[feature[i]]
            ### post-processing
            x_pp_dp = eval_pp_dp[feature[j]]
            y_pp_dp = eval_pp_dp[feature[i]]
            x_pp_eq_odds = eval_pp_eq_odds[feature[j]]
            y_pp_eq_odds = eval_pp_eq_odds[feature[i]]
            ### fair log-loss
            x_fairll_dp = eval_fairll_dp[feature[j]]
            y_fairll_dp = eval_fairll_dp[feature[i]]
            x_fairll_eq_odds = eval_fairll_eqodds[feature[j]]
            y_fairll_eq_odds = eval_fairll_eqodds[feature[i]]
            ### FairIRL Bias Reduction (no point when the session did not run it)
            x_fair_irl = [] if eval_fair_irl is None else [eval_fair_irl[feature[j]]]
            y_fair_irl = [] if eval_fair_irl is None else [eval_fair_irl[feature[i]]]

            newX = x + alpha[j]
            newY = y + alpha[i]
            xlim = max(max(demo_metric_j), x_fairll_eq_odds, x_fairll_dp, x_pp_eq_odds, x_pp_dp, *x_fair_irl, newX) * 1.2
            ylim = max(max(demo_metric_i), y_fairll_eq_odds, y_fairll_dp, y_pp_eq_odds, y_pp_dp, *y_fair_irl, newY) * 1.2
            plt.xlabel(name[feature[j]])
            plt.ylabel(name[feature[i]])

            plt.plot(x_test, y_test, "Xk", label="superhuman_test")
            plt.plot(x, y, "ro", label="superhuman_train")
            plt.scatter(demo_metric_j, demo_metric_i, marker="*", c="orange", label=demo_label)
            # plot FairIRL Bias Reduction
            if eval_fair_irl is not None:
                plt.plot(x_fair_irl[0], y_fair_irl[0], "hm", label="fair_irl_bias_reduction")
            # plot post-processing
            plt.plot(x_pp_dp, y_pp_dp, "bo", label="post_proc_dp")
            plt.plot(x_pp_eq_odds, y_pp_eq_odds, "go", label="post_proc_eqodds")
            # plot fair log-loss
            plt.plot(x_fairll_dp, y_fairll_dp, marker="P", color="darkcyan", label="fair_logloss_dp")
            plt.plot(x_fairll_eq_odds, y_fairll_eq_odds, marker="P", color="indigo", label="fair_logloss_eqodds")

            xmin, xmax, ymin, ymax = plt.axis()

            yLeft = (ylim - y) * 0.8 + y
            xBottom = (xlim - x) * 0.8 + x
            plt.plot([x, x], [y, ylim], "r")
            plt.plot([x, xlim], [y, y], "r")
            plt.plot([newX, newX], [newY, ylim], "r--")
            plt.plot([newX, xlim], [newY, newY], "r--")
            plt.annotate("", xy=(newX, yLeft), xytext=(x, yLeft), xycoords="data", textcoords="data",
                         arrowprops={"arrowstyle": "<->"})
            # write the text to the top of the arrow above
            plt.text((newX + x) * 0.5, yLeft, fr"$1/\alpha_{{{short[feature[j]]}}}$", horizontalalignment="center", verticalalignment="bottom")
            plt.annotate("", xy=(xBottom, newY), xytext=(xBottom, y), xycoords="data", textcoords="data",
                         arrowprops={"arrowstyle": "<->"})
            # write the text to the right of the arrow above
            plt.text(xBottom, (newY + y) * 0.5,
                     fr"$1/\alpha_{{{short[feature[i]]}}}$", horizontalalignment="left", verticalalignment="center")

            handles, labels = plt.gca().get_legend_handles_labels()
            plt.grid(True)
            plt.legend(reversed(handles), reversed(labels), loc="best", ncol=1, fontsize="small")
            plt.title(dataset)
            plot_file_name = short[feature[j]] + "_vs_" + short[feature[i]] + "_{}_{}_{}".format(dataset, expert, bias_type_label(dataset_bias_type)).replace(".", "-") + ".pdf"
            plots_path_dir = os.path.join(plots_path, plot_file_name)
            plt.savefig(plots_path_dir)
            plt.show()
            plt.close(f1)


for dataset, expert, dataset_bias_type in plotted_configurations():
    plot_features(
        dataset,
        expert,
        dataset_bias_type,
        resolve_fairirl_weight_adjust(
            dataset, expert, dataset_bias_type, plot_fairirl_weight_adjust
        ),
    )

In [ ]:
# plot_features_errorbars: `plot_features_errorbars()` of the Superhuman
# Fairness reference implementation's `plot.py`. Run the setup cell above first.
#
# The same figures as `plot_features`, but every model is drawn at its mean
# over the session's trials (`per_trial_data`), with error bars of
# std * 1.96 / sqrt(n_trials) -- a 95% confidence interval -- while the
# Superhuman Fairness training point, its 1/alpha margins and the
# demonstrations are those of the single trial `exp_idx`.
#
# Beyond the changes listed in the setup cell:
# * `baselines` is iterated in a fixed order; the original iterated a set, whose
#   order (and so the legend's) changes between Python processes.
# * Each model's confidence interval uses its own number of trials, rather than
#   one global experiment count, so a trial whose run failed only narrows that
#   model's interval instead of breaking the figure.
# * The original borrowed MFOpt's error bars from post-processing EqOdds, as it
#   had no trials of its own; that goes with MFOpt.
#
# With N_TRIALS = 1 every error bar has zero width.

# The trial whose training point, margins and demonstrations are drawn. The
# original used 4, of its 10 experiments.
exp_idx = 0

baselines = (
    "eval_sh",
    "eval_pp_dp",
    "eval_pp_eq_odds",
    "eval_fairll_dp",
    "eval_fairll_eqodds",
    "eval_fair_irl",
)
marker = {"eval_sh": "P", "eval_pp_dp": "o", "eval_pp_eq_odds": "o", "eval_fairll_dp": "P", "eval_fairll_eqodds": "P", "eval_fair_irl": "h"}
color = {"eval_sh": "k", "eval_pp_dp": "b", "eval_pp_eq_odds": "g", "eval_fairll_dp": "darkcyan", "eval_fairll_eqodds": "indigo", "eval_fair_irl": "m"}
label = {"eval_sh": "superhuman_test", "eval_pp_dp": "post_proc_dp", "eval_pp_eq_odds": "post_proc_eqodds", "eval_fairll_dp": "fair_logloss_dp", "eval_fairll_eqodds": "fair_logloss_eqodds", "eval_fair_irl": "fair_irl_bias_reduction"}
# The fairness constraint of each fair log-loss model, for `fair_logloss_losses()`.
fair_logloss_constraint = {"eval_fairll_dp": "demographic_parity", "eval_fairll_eqodds": "equalized_odds"}


def test_losses(method, results):
    """A model's test losses, with `fair_logloss_losses()` for the fair
    log-loss models."""
    if method in fair_logloss_constraint:
        return fair_logloss_losses(results, "test", fair_logloss_constraint[method])
    return feature_losses(results, "test")


def plot_features_errorbars(dataset, expert, dataset_bias_type, fairirl_weight_adjust, exp_idx):
    by_algorithm = per_trial_data[dataset][expert]
    sh_trials = by_algorithm["Superhuman Fairness"][dataset_bias_type][()]
    # Each model's results, one row per trial.
    method_trials = {
        "eval_sh": sh_trials,
        "eval_pp_dp": by_algorithm["Post Proc DP"][dataset_bias_type][()],
        "eval_pp_eq_odds": by_algorithm["Post Proc EqOdds"][dataset_bias_type][()],
        "eval_fairll_dp": by_algorithm["Fair LogLoss DP"][dataset_bias_type][()],
        "eval_fairll_eqodds": by_algorithm["Fair LogLoss EqOdds"][dataset_bias_type][()],
    }
    # FairIRL Bias Reduction, if the session ran it for this configuration
    if fairirl_weight_adjust is not None:
        method_trials["eval_fair_irl"] = by_algorithm["FairIRL Bias Reduction"][dataset_bias_type][fairirl_weight_adjust]
    methods = tuple(method for method in baselines if method in method_trials)
    if exp_idx not in sh_trials.index:
        raise KeyError(
            f"exp_idx={exp_idx} is not a trial of {dataset} / {expert} /"
            f" {bias_type_label(dataset_bias_type)}; its trials are"
            f" {list(sh_trials.index)}."
        )

    num_experiment = len(sh_trials)
    demo_losses, demo_label = get_plot_demo_losses(
        dataset, expert, dataset_bias_type, trial=exp_idx
    )
    print(dataset, expert, bias_type_label(dataset_bias_type))
    alpha = superhuman_alpha(sh_trials.loc[exp_idx])
    print("alpha: ", alpha)
    margin = [1 / x for x in alpha]
    print("margin: ", margin)
    print(len(sh_trials))
    print("num_experiment: ", num_experiment)

    # The original reports these diagnostics for its first experiment.
    first_trial = sh_trials.index[0]
    find_gamma_superhuman(demo_losses, feature_losses(sh_trials.loc[first_trial], "train"))
    find_gamma_superhuman_all(
        demo_losses,
        {
            **{
                method: test_losses(method, method_trials[method].loc[first_trial])
                for method in methods
                if method != "eval_sh"
            },
            "superhuman": feature_losses(sh_trials.loc[first_trial], "train"),
        },
    )

    ### our model, on its training split, for the trial `exp_idx`
    eval_train = feature_losses(sh_trials.loc[exp_idx], "train")
    # Every model's per-trial losses on the test split
    method_losses = {method: test_losses(method, method_trials[method]) for method in methods}

    for i in range(num_of_features):
        for j in range(i + 1, num_of_features):
            demo_metric_i = demo_losses[feature[i]].tolist()
            demo_metric_j = demo_losses[feature[j]].tolist()
            f1 = plt.figure()
            ### our model
            x = eval_train[feature[j]]
            y = eval_train[feature[i]]
            plts_data = {}
            for method in methods:
                std_coef = 1.96 / np.sqrt(len(method_trials[method]))
                plts_data[method] = {}
                plts_data[method]["x"], plts_data[method]["y"] = [], []
                for k in method_trials[method].index:
                    plts_data[method]["x"].append(method_losses[method][feature[j]].loc[k])
                    plts_data[method]["y"].append(method_losses[method][feature[i]].loc[k])

                plts_data[method]["x_mean"] = np.mean(plts_data[method]["x"])
                plts_data[method]["y_mean"] = np.mean(plts_data[method]["y"])
                plts_data[method]["x_err"] = np.std(plts_data[method]["x"]) * std_coef
                plts_data[method]["y_err"] = np.std(plts_data[method]["y"]) * std_coef

            newX = x + margin[j]
            newY = y + margin[i]

            xlim = max(max(demo_metric_j), max([plts_data[method]["x_mean"] for method in methods]), newX) * 1.2
            ylim = max(max(demo_metric_i), max([plts_data[method]["y_mean"] for method in methods]), newY) * 1.2

            plt.xlabel(name[feature[j]])
            plt.ylabel(name[feature[i]])

            plt.plot(x, y, "ro", label="superhuman_train")

            plt.scatter(demo_metric_j, demo_metric_i, marker="*", c="orange", label=demo_label)

            for method in methods:
                plt.errorbar(plts_data[method]["x_mean"], plts_data[method]["y_mean"], xerr=plts_data[method]["x_err"], yerr=plts_data[method]["y_err"], marker=marker[method], color=color[method], label=label[method])
                print()
                print("{}: ".format(method))
                print("{}: {} + {}".format(feature[j], plts_data[method]["x_mean"], plts_data[method]["x_err"]))
                print("{}: {} + {}".format(feature[i], plts_data[method]["y_mean"], plts_data[method]["y_err"]))
                print()

            xmin, xmax, ymin, ymax = plt.axis()

            yLeft = (ylim - y) * 0.8 + y
            xBottom = (xlim - x) * 0.8 + x
            plt.plot([x, x], [y, ylim], "r")
            plt.plot([x, xlim], [y, y], "r")
            plt.plot([newX, newX], [newY, ylim], "r--")
            plt.plot([newX, xlim], [newY, newY], "r--")
            plt.annotate("", xy=(newX, yLeft), xytext=(x, yLeft), xycoords="data", textcoords="data",
                         arrowprops={"arrowstyle": "<->"})
            # write the text to the top of the arrow above
            plt.text((newX + x) * 0.5, yLeft, fr"$1/\alpha_{{{short[feature[j]]}}}$", horizontalalignment="center", verticalalignment="bottom")
            plt.annotate("", xy=(xBottom, newY), xytext=(xBottom, y), xycoords="data", textcoords="data",
                         arrowprops={"arrowstyle": "<->"})
            # write the text to the right of the arrow above
            plt.text(xBottom, (newY + y) * 0.5,
                     fr"$1/\alpha_{{{short[feature[i]]}}}$", horizontalalignment="left", verticalalignment="center")

            handles, labels = plt.gca().get_legend_handles_labels()
            plt.grid(True)
            plt.legend(reversed(handles), reversed(labels), loc="best", ncol=1, fontsize="small")
            plt.title(dataset)
            plot_file_name = "errbar_" + short[feature[j]] + "_vs_" + short[feature[i]] + "_{}_{}_{}".format(dataset, expert, bias_type_label(dataset_bias_type)).replace(".", "-") + ".pdf"
            plots_path_dir = os.path.join(plots_path, plot_file_name)
            plt.savefig(plots_path_dir)
            plt.show()
            plt.close(f1)


for dataset, expert, dataset_bias_type in plotted_configurations():
    plot_features_errorbars(
        dataset,
        expert,
        dataset_bias_type,
        resolve_fairirl_weight_adjust(
            dataset, expert, dataset_bias_type, plot_fairirl_weight_adjust
        ),
        exp_idx,
    )

In [ ]:
unbiased_types = ()
# biased_types = ("perfectly_balanced_redlining", 0.2)
# biased_types = ("balanced_redlining", 0.2)
# biased_types = ("unbalanced_redlining", 0.2)
# biased_types = ("threshold_swapping", 0.2)
# biased_types = ("corruption_bias", "CatBoost", 0.001, "gaussian", 1.0)

unadjusted_weights = ()
# adjusted_weights = ("mul_negative_weights", 0.0)
# adjusted_weights = ("opt_debias", "optuna", "CMA-ES", 500)
# adjusted_weights = ("opt_debias", "pybobyqa", "Multi-Start BOBYQA", 500)
adjusted_weights = ("opt_debias", "nevergrad", "BayesOpt", 200)
# adjusted_weights = ("opt_debias", "nevergrad", "Nelder-Mead", 500)
# adjusted_weights = ("opt_debias", "nevergrad", "Powell", 500)


# diff_metric_df_list = []

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muL_best_", "", biased_types, unadjusted_weights))  # Original IRL FE
data_config_list.append(
    ("muL_test_", "", biased_types, adjusted_weights)
)  # Zeroed IRL FE
measurable_metrics = [
    "Acc",
    "AccPar",
    "DemPar",
    "EqOpp",
    "TNRPar",
    # "EqOdds"
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

# selected_expert = "OptClfMDPPol"
diff_metric_list = []
dataset_list = []
for dataset in averaged_info:
    dataset_list.append(dataset)
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][selected_algorithm][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[0]
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
# diff_metric_df_list.append(diff_metric_df)
df1 = diff_metric_df

data_config_list = []  # (Metric Prefix, Metrix Suffix, bias types, weight adjustments)
# data_config_list.append(("muE_", "_mean", biased_types, unadjusted_weights)) # Biased Demo FE
data_config_list.append(
    ("muE_test_unbiased_", "_mean", unbiased_types, unadjusted_weights)
)  # Raw Data FE
measurable_metrics = [
    "Acc",
    "AccPar",
    "DemPar",
    "EqOpp",
    "TNRPar",
    # "EqOdds"
]  # , "FPRPar", "FNRPar"] # removed FPRPar and FNRPar because they are effectively the same as EqOpp and TNRPar
measurable_metric_list = []
for data_config in data_config_list:
    measurable_metric_list.append(
        [data_config[0] + metric + data_config[1] for metric in measurable_metrics]
    )

# selected_expert = "OptClfMDPPol"
diff_metric_list = []
for dataset in averaged_info:
    metric_series_list = []
    for data_config, measurable_metric in zip(data_config_list, measurable_metric_list):
        exp_results = averaged_data[dataset][selected_expert][selected_algorithm][data_config[2]][
            data_config[3]
        ]
        metric_results = exp_results.loc[measurable_metric]
        metric_results.index = measurable_metrics
        metric_series_list.append(metric_results)
    diff_sub_series = metric_series_list[0]
    diff_metric_list.append(diff_sub_series)
diff_metric_df = pd.concat(diff_metric_list, axis=1).T
diff_metric_df.index = dataset_list
# diff_metric_df_list.append(diff_metric_df)
df2 = diff_metric_df

# -------------------------------------------------------------------
# PLOT CONFIGURATION
# -------------------------------------------------------------------
n_rows = len(df1)  # number of samples per column
n_cols = len(df1.columns)  # number of metrics/columns
colors = {"df1": "tab:blue", "df2": "tab:red"}  # distinct colors
alpha = 0.5  # transparency (0 = fully transparent, 1 = opaque)
bar_width = 0.8  # width of each bar
gap_between_columns = 1.0  # horizontal gap BETWEEN metric groups
label_offset = (
    0.05  # Space between lowest bar bottom and label (as fraction of bar height)
)

fig, ax = plt.subplots(figsize=(18, 7))

# -------------------------------------------------------------------
# PLOT BARS
# -------------------------------------------------------------------
for col_idx, col_name in enumerate(df1.columns):

    # Start position for this metric group on the x‑axis
    start_x = col_idx * (n_rows + gap_between_columns)

    # Extract the values for this column from BOTH DataFrames
    vals_df1 = df1[col_name].values
    vals_df2 = df2[col_name].values

    # Plot every sample (row) in this column
    for row_idx in range(n_rows):
        x_pos = start_x + row_idx

        # Get the row label (index name)
        row_label = df1.index[row_idx]

        # Current values for both bars
        val1 = vals_df1[row_idx]
        val2 = vals_df2[row_idx]

        # -------------------------------------------------------------------
        # CALCULATE SAFE LABEL POSITION (Handles Positive & Negative Data)
        # -------------------------------------------------------------------
        # Find the absolute bottom (most negative value) between the two bars
        lowest_bottom = min(val1, val2, 0.0)

        # Determine offset distance based on bar height to keep proportional spacing
        # Use max absolute height to ensure enough space regardless of bar size
        max_height = max(abs(val1), abs(val2))
        padding_distance = max(0.01 * max_height, 0.01)

        # Calculate final y-position for label (below the lowest bar)
        label_y = lowest_bottom - padding_distance

        # Bar for DataFrame 1
        ax.bar(
            x_pos,
            val1,
            width=bar_width,
            alpha=alpha,
            color=colors["df1"],
            edgecolor="black",
        )

        # Bar for DataFrame 2 (OVERLAPS with DF1 at the SAME x_pos!)
        ax.bar(
            x_pos,
            val2,
            width=bar_width,
            alpha=alpha,
            color=colors["df2"],
            edgecolor="black",
        )

        # Add Row Index Label Below the Bars (Safe for Negative Data)
        ax.text(
            x_pos,
            label_y,
            row_label,
            ha="center",
            va="top",
            fontsize=9,
            fontweight="bold",
            rotation=90,
            color="darkgray",
            bbox=dict(facecolor="white", edgecolor="none", boxstyle="round,pad=0.2"),
        )

# -------------------------------------------------------------------
# X‑AXIS LABELS (one tick per metric group)
# -------------------------------------------------------------------
# Place a tick at the *center* of each metric group
tick_positions = [
    col_idx * (n_rows + gap_between_columns) + (n_rows - 1) / 2
    for col_idx in range(n_cols)
]
ax.set_xticks(tick_positions)
ax.set_xticklabels(df1.columns, fontsize=11, fontweight="bold", rotation=90)

# -------------------------------------------------------------------
# FINAL TOUCHES
# -------------------------------------------------------------------
ax.set_ylabel("Feature Expectation", fontsize=11)
ax.set_title(
    f"Comparison of How Feature Expectations Changed for {adjusted_weights}",
    fontsize=13,
    fontweight="bold",
)
ax.legend(
    ["Feature Expectation for Adjusted Weights", "Feature Expectation Before Bias Added"],
    loc="upper right",
    framealpha=0.5,
    fancybox=True,
)
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig(f"./../../experiment_output/fair_irl/exp_plots/Feat_Exp_Before_After.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Visualize what happens to the feature expectations throughout the training process if you fully zero the negative weights for each dataset. Show that if you just zero the negative weights, you get something where the feature expectations dont match what the expert originally was attempting
# feature expectations of the dataset with no bias
# feature expectations of the dataset with bias added
# feature expectations of Fair-IRL trained on bias with no zeroing
# Feature expectations of Fair-IRL trained on bias with zeroing

# selected_expert = "OptClfMDPPol"
unbiased_type = ()
# biased_type = ("unbalanced_redlining", 0.2)
# biased_type = ("balanced_redlining", 0.2)
# biased_type = ("perfectly_balanced_redlining", 0.2)
biased_type = ("corruption_bias", "CatBoost", 0.001, "gaussian", 1.0)

all_datasets = [
    "COMPAS",
    # "Boston",
    "Adult",
    "ACSIncome__MA",
    "ACSIncome__MS",
    "ACSIncome__CA",
    "ACSIncome__IL",
    "ACSIncome__AL",
    "ACSIncome__HI",
]

measurable_metrics = [
    "Acc",
    "AccPar",
    "DemPar",
    "EqOpp",
    "TNRPar",
]

muE_test_unbiased_names = ["muE_test_unbiased_" + metric + "_mean" for metric in measurable_metrics]
muE_test_names = ["muE_test_" + metric + "_mean" for metric in measurable_metrics]
muL_test_names = ["muL_test_" + metric for metric in measurable_metrics]

unadjusted_weights = ()
# adjusted_weights = ("mul_negative_weights", 0.0)
adjusted_weights = ("opt_debias", "nevergrad", "BayesOpt", 200)

dataset_results = {}

for dataset in all_datasets:
    muE_test_unbiased = averaged_data[dataset][selected_expert][selected_algorithm][unbiased_type][unadjusted_weights][muE_test_unbiased_names].values.tolist()
    
    muE_test = averaged_data[dataset][selected_expert][selected_algorithm][biased_type][unadjusted_weights][muE_test_names].values.tolist()

    muL_test_unbiased = averaged_data[dataset][selected_expert][selected_algorithm][unbiased_type][unadjusted_weights][muL_test_names].values.tolist()

    muL_test_unadjusted = averaged_data[dataset][selected_expert][selected_algorithm][biased_type][unadjusted_weights][muL_test_names].values.tolist()

    muL_test_zeroed = averaged_data[dataset][selected_expert][selected_algorithm][biased_type][adjusted_weights][muL_test_names].values.tolist()
    
    dataset_results[dataset] = (muE_test_unbiased, muE_test, muL_test_unadjusted, muL_test_zeroed)

# Plotting
for dataset, (muE_test_unbiased, muE_test, muL_test_unadjusted, muL_test_zeroed) in dataset_results.items():
    
    # Bar settings - FIXED: Use 4 bars per cluster with proper positioning
    x = np.arange(len(measurable_metrics))  # x positions for metrics
    width = 0.15  # width of each bar (narrower to fit 4 bars)
    
    fig, ax = plt.subplots(figsize=(16, 6))

    colors_muE_test_unbiased = "red"
    colors_muE_test = "yellow"
    colors_muL_test_unbiased = "orange"
    colors_muL_test_unadjusted = "green"
    colors_muL_test_zeroed = "blue"

    # Clustered bars - FIXED: Proper positioning for 4 bars
    # Calculate offsets so bars are evenly spaced around each x position
    ax.bar(x - 1.5*width, muE_test_unbiased, width, 
           label=f'{selected_expert} Expert on Unbiased Data', color=colors_muE_test_unbiased)
    ax.bar(x - 0.5*width, muE_test, width, 
           label=f'{selected_expert} Expert on Data With {biased_type[0]} Added', color=colors_muE_test)
    ax.bar(x + 0.5*width, muL_test_unbiased, width, 
           label='FairIRL Bias Reduction Model With No Weight Adjustment and No Bias', color=colors_muL_test_unbiased)
    ax.bar(x + 1.5*width, muL_test_unadjusted, width, 
           label=f'FairIRL Bias Reduction Model With No Weight Adjustment on Data With {biased_type[0]} Added', color=colors_muL_test_unadjusted)
    ax.bar(x + 2.5*width, muL_test_zeroed, width, 
           label=f'FairIRL Bias Reduction Model With Weight Adjustment on Data With {biased_type[0]} Added', color=colors_muL_test_zeroed)

    # Labels and title
    ax.set_ylabel('Learned Feature Expectations')
    ax.set_title(f"Learned Feature Expectations Throughout the FairIRL Bias Reduction Training Pipeline for the {dataset} Dataset")
    ax.set_xticks(x)
    ax.set_xticklabels(measurable_metrics, rotation=45, ha='right')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.legend()
    
    # Add grid for better readability
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    
    # Save and show
    plt.savefig(f"./../../experiment_output/fair_irl/exp_plots/Learned_Feat_Exp_of_Train_Pipeline_{dataset}.pdf", bbox_inches="tight")
    plt.show()